# 08 · From text to map, without losing the text

**Spatial Humanities 2026 workshop**

Maps are powerful summaries, but not every spatial expression has a defensible coordinate and not every coordinate should be displayed without uncertainty. This notebook converts entity and journey records into spatial representations while preserving an audit trail for what was omitted or uncertain.

## Learning goals
- export linked entities as GeoJSON points;
- export structured journeys as GeoJSON routes;
- inspect routes that were skipped because endpoints were missing, unresolved or ambiguous;
- compare textual nearness with Euclidean distance;
- create a simple co-occurrence representation;
- explain why mapping is one representation of spatial evidence, not its endpoint.

> **Key message:** A map should expose its omissions and uncertainties, not conceal them.

In [ ]:
# Independent Colab setup.
import os, sys, json, subprocess, pathlib

REPO = "https://github.com/IgnatiusEzeani/spatio-textual.git"
BRANCH = "spatial-humanities-2026"

if not pathlib.Path("spatio-textual").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "spatio-textual"], check=True)
os.chdir("spatio-textual")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[app]"], check=True)

print("Ready:", pathlib.Path.cwd())

## 1. Point features from linked entities

`to_geojson()` includes only entities that have usable coordinates. Original entity strings and provenance remain in feature properties.

In [ ]:
from spatio_textual.geocode import GeoResolver
from spatio_textual.viz import to_geojson

resolver = GeoResolver()

entities = []
for name in ["London", "Paris", "Amsterdam"]:
    resolved = resolver.resolve(name, label="GPE", context=f"The text mentions {name}.")
    entities.append({"text": name, "label": "GPE", "place_type": "PLACE", **resolved})

record = {"fileId": "map-demo-01", "segId": 0, "entities": entities}
point_geojson = to_geojson([record])
print(json.dumps(point_geojson, indent=2, ensure_ascii=False))

## 2. Route features from journey records

The SH2026 route exporter is deliberately conservative. It maps a route only when both endpoints are resolvable. Ambiguous endpoints are skipped by default and recorded in an `audit` member.

In [ ]:
from spatio_textual.viz import journeys_to_geojson

journeys = [
    {
        "journeyId": "j-london-paris", "fileId": "map-demo-01", "segId": 0,
        "start_location": "London", "end_location": "Paris", "transport_mode": "train",
        "date": None, "journey_reason": None,
        "evidence_quote": "I travelled from London to Paris by train.",
        "explicit_or_inferred": {
            "start_location": "explicit", "end_location": "explicit", "transport_mode": "explicit",
            "date": "missing", "journey_reason": "missing",
        },
        "confidence": 0.95, "requires_review": False,
    },
    {
        "journeyId": "j-cambridge-london", "fileId": "map-demo-01", "segId": 1,
        "start_location": "Cambridge", "end_location": "London",
        "evidence_quote": "I travelled from Cambridge to London.",
        "requires_review": True,
    },
    {
        "journeyId": "j-historical", "fileId": "map-demo-01", "segId": 2,
        "start_location": "Czechoslovakia", "end_location": "London",
        "evidence_quote": "I travelled from Czechoslovakia to London.",
        "requires_review": True,
    },
]

route_geojson = journeys_to_geojson(journeys, resolver=resolver, allow_ambiguous=False)
print(json.dumps(route_geojson, indent=2, ensure_ascii=False))

The `audit` member is as important as the route itself. It tells us **which journeys did not become lines on the map and why**.

In [ ]:
import pandas as pd

display(pd.DataFrame(route_geojson["audit"]))

## 3. What changes if ambiguity is allowed?

For demonstration, we can permit ambiguous endpoints. This does not make the ambiguity disappear: the route properties still record that review is required.

In [ ]:
route_geojson_permissive = journeys_to_geojson(journeys, resolver=resolver, allow_ambiguous=True)
display(pd.DataFrame([
    {
        "journeyId": f["properties"]["journeyId"],
        "resolved_start": f["properties"]["resolved_start"],
        "resolved_end": f["properties"]["resolved_end"],
        "ambiguous_endpoint": f["properties"]["ambiguous_endpoint"],
        "requires_review": f["properties"]["requires_review"],
    }
    for f in route_geojson_permissive["features"]
]))

A permissive map may be useful for exploration, but it should not be exported as if all routes had equal evidential status.

## 4. Render an interactive map

The same GeoJSON can be saved as an interactive Folium map. The notebook displays the map locally; the Streamlit demo will later render this directly rather than exposing raw GeoJSON.

In [ ]:
from spatio_textual.viz import make_map_geojson
from IPython.display import IFrame, display
from pathlib import Path

out_dir = Path("sh2026_outputs/geojson")
out_dir.mkdir(parents=True, exist_ok=True)
map_path = make_map_geojson(route_geojson_permissive, out_dir / "journey_routes.html")
display(IFrame(src=map_path, width="100%", height=500))

## 5. Textual nearness is not Euclidean nearness

Spatial narratives can describe places as close, remote, beyond, nearby or familiar for rhetorical and experiential reasons. A physical distance is useful context, but it does not replace the textual relation.

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine_km(lat1, lon1, lat2, lon2):
    radius = 6371.0088
    p1, p2 = radians(lat1), radians(lat2)
    dp = radians(lat2 - lat1)
    dl = radians(lon2 - lon1)
    a = sin(dp/2)**2 + cos(p1) * cos(p2) * sin(dl/2)**2
    return 2 * radius * atan2(sqrt(a), sqrt(1-a))

london = resolver.resolve("London", label="GPE")
paris = resolver.resolve("Paris", label="GPE")
distance = haversine_km(london["lat"], london["lon"], paris["lat"], paris["lon"])

synthetic_relation = "In the story, Paris felt near to London because the journey linked the two places repeatedly."
print(synthetic_relation)
print(f"Approximate straight-line distance: {distance:.1f} km")

The statement above is intentionally synthetic. Its purpose is to show that **textual nearness and metric distance answer different questions**.

For the Lake District case study, we will later use genuine source/corpus examples and verified coordinates to compare textual co-occurrence/nearness with physical proximity.

## 6. Co-occurrence as another spatial representation

Maps are not the only way to represent place relations. Co-occurrence can reveal which named places are narratively connected within the same analytical unit.

In [ ]:
from spatio_textual.viz import build_cooccurrence

records = [
    {"entities": [{"text": "London", "label": "GPE"}, {"text": "Paris", "label": "GPE"}]},
    {"entities": [{"text": "London", "label": "GPE"}, {"text": "Amsterdam", "label": "GPE"}]},
    {"entities": [{"text": "London", "label": "GPE"}, {"text": "Paris", "label": "GPE"}]},
]

edges = build_cooccurrence(records)
display(pd.DataFrame(edges, columns=["source", "target", "weight"]))

A co-occurrence edge means the entities appeared together under the chosen analytical window. It does **not** by itself mean travel, causality, physical proximity or social connection.

## 7. What should not be forced onto a point map?

Consider:

- `home`
- `the village`
- `beyond the river`
- `to our left`
- `near the old road`
- a historical polity without time-aware geometry

These expressions can carry substantial spatial meaning while lacking a single defensible modern coordinate.

A responsible system should preserve them as textual/spatial evidence rather than dropping them silently or inventing a point.

## 8. Export both representation and audit trail

In [ ]:
geojson_path = out_dir / "journey_routes.geojson"
with geojson_path.open("w", encoding="utf-8") as fh:
    json.dump(route_geojson_permissive, fh, ensure_ascii=False, indent=2)

print(geojson_path)
print("Mapped routes:", len(route_geojson_permissive["features"]))
print("Audit records:", len(route_geojson_permissive["audit"]))

## 9. Take-away

A spatial visualisation is itself an analytical transformation:

**text → recognition → resolution → relation → coordinate/geometry → visual encoding**

Every arrow can introduce interpretation. The scholarly map should therefore remain connected to source evidence, uncertainty and omitted cases.

**Next:** consolidate these principles into a reusable Responsible Spatial AI checklist and release audit.